## Main code of Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class MPM_Node(nn.Module):
    """
    We have implemented a ConvLSTM cell as the main the module of the Motion Pattern Module (MPM).
    """
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super(MPM_Node, self).__init__()
        self.padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size, padding=self.padding
        )

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.split(gates, gates.size(1) // 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c_prev + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class TemporalSelfAttention(nn.Module):
    """
    Multi-head self-attention across the temporal dimension implemented from scratch.
    Rather than using attention across spatial locations, we attend across the T hidden states at each (h, w) position. 

    It operates on [B, C, T, H, W] by flattening spatial dims into the
    sequence so each (b, h, w) position attends over its T hidden states
    """
    def __init__(self, channels: int, num_heads: int = 4, dropout: float = 0.0):
        super(TemporalSelfAttention, self).__init__()
        assert channels % num_heads == 0, (
            f"channels ({channels}) must be divisible by num_heads ({num_heads})"
        )
        self.num_heads = num_heads
        self.head_dim  = channels // num_heads
        self.scale     = math.sqrt(self.head_dim)

        self.qkv_proj  = nn.Linear(channels, 3 * channels, bias=False)
        self.out_proj  = nn.Linear(channels, channels, bias=False)
        self.dropout   = nn.Dropout(dropout)
        self._pos_bias: dict = {}

    def _get_pos_bias(self, T: int, device: torch.device) -> torch.Tensor:
        if T not in self._pos_bias:
            bias = nn.Parameter(torch.zeros(T, T, device=device))
            nn.init.trunc_normal_(bias, std=0.02)
            self._pos_bias[T] = bias
            self.register_parameter(f"pos_bias_T{T}", bias)
        return self._pos_bias[T].to(device)

    def forward(self, h_s: torch.Tensor) -> torch.Tensor:
        """
        Args:  h_s: [B, C, T, H, W]
        Returns:    [B, C, T, H, W]
        """
        B, C, T, H, W = h_s.shape
        x   = h_s.permute(0, 3, 4, 2, 1).reshape(B * H * W, T, C)
        qkv = self.qkv_proj(x).reshape(B * H * W, T, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) / self.scale
        pos_bias = self._get_pos_bias(T, h_s.device)
        attn = attn + pos_bias.unsqueeze(0).unsqueeze(0)
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).permute(0, 2, 1, 3).reshape(B * H * W, T, C)
        out = self.out_proj(out) + x                          # residual
        out = out.reshape(B, H, W, T, C).permute(0, 4, 3, 1, 2)
        return out


class MotionVisionAdapter(nn.Module):
    """Reduces the semantic gap between vision and motion patterns."""
    def __init__(self, channels, T):
        super(MotionVisionAdapter, self).__init__()
        self.T = T
        self.relation_conv = nn.Conv2d(2 * channels, channels // T, kernel_size=1)
        self.adapting_conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, h_tilde, h_sequence):
        outputs = []
        for t in range(self.T):
            h_t      = h_sequence[:, :, t, :, :]
            combined = torch.cat([h_tilde, h_t], dim=1)
            relation_weight = F.relu(self.relation_conv(combined))
            adapted  = self.adapting_conv(h_t * relation_weight.mean(dim=1, keepdim=True))
            outputs.append(adapted)
        return torch.stack(outputs, dim=2)


class MICPL_Module(nn.Module):
    """
    We pass the data in the following order: MPM → TemporalSelfAttention → MVA
    1. MPM  (ConvLSTM)            — recurrent hidden states over T time steps
    2. TemporalSelfAttention      — each spatial location attends over all T states
    3. MVA                        — motion-vision adaptation using last attended state to adapt each frame's features
    """
    def __init__(self, channels, T, num_heads: int = 4, num_layers: int = 2):
        super(MICPL_Module, self).__init__()
        self.T          = T
        self.num_layers = num_layers

        self.mpm_layers   = nn.ModuleList([
            MPM_Node(channels, channels) for _ in range(num_layers)
        ])
        self.temporal_attn = TemporalSelfAttention(channels, num_heads=num_heads)
        self.mva           = MotionVisionAdapter(channels, T)

    def forward(self, x_s: torch.Tensor) -> torch.Tensor:
        batch, channels, T, height, width = x_s.size()

        h = [torch.zeros(batch, channels, height, width, device=x_s.device)
             for _ in range(self.num_layers)]
        c = [torch.zeros(batch, channels, height, width, device=x_s.device)
             for _ in range(self.num_layers)]

        layer_outputs = []


        for t in range(T):
            current_input = x_s[:, :, t, :, :]
            for l in range(self.num_layers):
                h[l], c[l] = self.mpm_layers[l](current_input, h[l], c[l])
                current_input = h[l]
            layer_outputs.append(h[-1])

        h_s = torch.stack(layer_outputs, dim=2)  

        h_s = self.temporal_attn(h_s)

        h_tilde        = h_s[:, :, -1, :, :]      
        motion_patterns = self.mva(h_tilde, h_s)
        return motion_patterns

class RAFTFlowEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
        self.raft = raft_small(weights=Raft_Small_Weights.DEFAULT)
        self.raft.requires_grad_(False)
        print("RAFT-small loaded (pretrained, frozen).")

    @torch.no_grad()
    def forward(self, frame1, frame2):
        f1 = (frame1 * 255.0).clamp(0, 255)
        f2 = (frame2 * 255.0).clamp(0, 255)
        flow_predictions = self.raft(f1, f2)
        flow = flow_predictions[-1].clone()
        del flow_predictions, f1, f2
        return flow   # [B, 2, H, W]


def warp_features(x, flow):
    """
    Warp feature map x [B, C, H, W] using full-res flow [B, 2, fH, fW].
    Flow is downsampled and rescaled to match the feature map resolution.
    """
    B, C, H, W = x.shape
    flow_down   = F.interpolate(flow, size=(H, W), mode='bilinear', align_corners=False)
    flow_down[:, 0] *= W / flow.shape[3]
    flow_down[:, 1] *= H / flow.shape[2]

    grid_y, grid_x = torch.meshgrid(
        torch.arange(H, device=x.device, dtype=torch.float32),
        torch.arange(W, device=x.device, dtype=torch.float32),
        indexing='ij'
    )
    grid     = torch.stack([grid_x, grid_y], dim=0).unsqueeze(0).expand(B, -1, -1, -1)
    new_grid = grid + flow_down
    new_grid[:, 0] = 2.0 * new_grid[:, 0] / (W - 1) - 1.0
    new_grid[:, 1] = 2.0 * new_grid[:, 1] / (H - 1) - 1.0
    new_grid = new_grid.permute(0, 2, 3, 1)   # [B, H, W, 2]
    return F.grid_sample(x, new_grid, mode='bilinear',
                         padding_mode='border', align_corners=True)


class SmallObjectDetector(nn.Module):

    def __init__(self, backbone, raft, channels=64, T=5, num_heads=4):
        super().__init__()
        self.backbone = backbone
        self.raft     = raft
        self.micpl    = MICPL_Module(channels, T, num_heads=num_heads)
        self.T        = T

    def forward(self, x_sequence, training=True):
        """
        Args:
            x_sequence: [B, 3, T, H, W]  raw RGB frames in [0, 1]
        """
        B, _, T, H, W = x_sequence.shape

        vision_features = []
        for t in range(T):
            feat = self.backbone(x_sequence[:, :, t, :, :])
            vision_features.append(feat)
        x_s = torch.stack(vision_features, dim=2)   

        flows = []
        for t in range(T - 1):
            flow = self.raft(x_sequence[:, :, t, :, :],
                             x_sequence[:, :, t + 1, :, :])
            flows.append(flow)

        x_s_warped_list = [x_s[:, :, 0, :, :]]     
        for t in range(1, T):
            warped = warp_features(x_s[:, :, t, :, :], flows[t - 1])
            x_s_warped_list.append(warped)
        x_s_warped = torch.stack(x_s_warped_list, dim=2)

        if training:
            h_s    = self.micpl(x_s_warped)
            result = x_s + h_s
            del flows, x_s_warped_list, x_s_warped, h_s
            return result
        else:
            del flows, x_s_warped_list, x_s_warped
            return x_s


In [ ]:

import torch
import torch.nn as nn
import timm


class DLA34FeatureExtractor(nn.Module):
    def __init__(self, out_channels=64, pretrained=True, freeze_backbone=True):
        super().__init__()
        DLA34_URL = "http://dl.yf.io/dla/models/imagenet/dla34-ba72cf86.pth"

        # self.dla = timm.create_model(
        #     'dla34',
        #     pretrained=False,
        #     features_only=True,
        #     out_indices=(1, 2, 3),
        # )

        # if pretrained:
        #     state_dict = torch.hub.load_state_dict_from_url(
        #         DLA34_URL, map_location='cpu', check_hash=False
        #     )
        #     self.dla.load_state_dict(state_dict, strict=False)
        #     print("DLA-34 weights loaded successfully.")
        self.dla=timm.create_model("hf_hub:timm/dla34.in1k", pretrained=pretrained,features_only=True,out_indices=(1,2,3))

        if freeze_backbone:
            self.dla.requires_grad_(False)
            print("DLA-34 backbone frozen.")

        channels = self.dla.feature_info.channels()
        c1, c2, c3 = channels

        self.up_c3 = nn.Conv2d(c3, out_channels, kernel_size=1, bias=False)
        self.up_c2 = nn.Conv2d(c2, out_channels, kernel_size=1, bias=False)
        self.up_c1 = nn.Conv2d(c1, out_channels, kernel_size=1, bias=False)

        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def unfreeze_backbone(self):
        self.dla.requires_grad_(True)
        print("Backbone unfrozen.")
    def forward(self, x):
        with torch.no_grad():
            f1, f2, f3 = self.dla(x)   

        target_size = f2.shape[-2:]

        p3 = F.interpolate(self.up_c3(f3), size=target_size, mode='bilinear', align_corners=False)
        p2 = self.up_c2(f2)
        p1 = F.avg_pool2d(self.up_c1(f1), 2)

        return self.fuse(p1 + p2 + p3)

## Dataset Code

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2


class VISOSequenceDataset(Dataset):
    def __init__(self, img_dir, ann_file, sequence_length=5, transform=None):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.sequence_length = sequence_length
        self.transform = transform

        self.img_ids = sorted(self.coco.getImgIds())
        self.valid_starts = len(self.img_ids) - self.sequence_length + 1

    def __len__(self):
        return self.valid_starts

    def _load_image(self, img_id):
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        image = cv2.imread(img_path)
        if image is None:
            h = img_info.get('height', 512)
            w = img_info.get('width', 512)
            return np.zeros((h, w, 3), dtype=np.uint8)
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    def _get_bboxes(self, img_id):

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns    = self.coco.loadAnns(ann_ids)
        bboxes, cat_ids = [], []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w <= 1 or h <= 1:
                continue
            x2, y2 = x + w, y + h
            if x2 <= x or y2 <= y:
                continue
            bboxes.append([x, y, x2, y2])
            cat_ids.append(ann['category_id'])
        return bboxes, cat_ids

    def __getitem__(self, idx):
        sequence_img_ids = self.img_ids[idx : idx + self.sequence_length]
        target_img_id    = sequence_img_ids[-1]   
        frames = [self._load_image(img_id) for img_id in sequence_img_ids]

        target_bboxes, target_cat_ids = self._get_bboxes(target_img_id)

        if self.transform:
            transformed   = self.transform(
                image=frames[-1],
                bboxes=target_bboxes,
                category_ids=target_cat_ids
            )
            replay_params = transformed['replay']

            new_frames = []
            for i in range(self.sequence_length - 1):
                replayed = A.ReplayCompose.replay(
                    replay_params,
                    image=frames[i],
                    bboxes=[],          
                    category_ids=[]
                )
                new_frames.append(replayed['image'])

            new_frames.append(transformed['image'])
            frames     = new_frames
            bboxes     = list(transformed['bboxes'])
            cat_ids    = list(transformed['category_ids'])

        else:
            frames  = [
                torch.from_numpy(f.transpose(2, 0, 1)).float() / 255.0
                for f in frames
            ]
            bboxes  = target_bboxes
            cat_ids = target_cat_ids

        sequence_tensor = torch.stack(frames, dim=1)

        target = {
            "bboxes": torch.tensor(bboxes,  dtype=torch.float32)
                      if bboxes else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.tensor(cat_ids, dtype=torch.int64)
                      if cat_ids else torch.zeros((0,),  dtype=torch.int64),
        }

        return sequence_tensor, target


def collate_fn(batch):
    sequences, targets = zip(*batch)
    return torch.stack(sequences, dim=0), list(targets)

transform = A.ReplayCompose([
    A.Resize(height=512, width=512),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], bbox_params=A.BboxParams(
    format='pascal_voc',
    label_fields=['category_ids'],
    clip=True,
    min_visibility=0.1,
    min_width=1.0,
    min_height=1.0,
))


IMG_DIR  = './VISO/coco/plane/train2017'
ANN_FILE = './VISO/coco/plane/Annotations/instances_train2017.json'

viso_dataset = VISOSequenceDataset(
    img_dir=IMG_DIR,
    ann_file=ANN_FILE,
    sequence_length=5,
    transform=transform,
)

data_loader = DataLoader(
    viso_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn,
)

In [ ]:
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import math

def gaussian_radius(det_size, min_overlap=0.3):
    height, width = det_size

    a1  = 1
    b1  = (height + width)
    c1  = width * height * (1 - min_overlap) / (1 + min_overlap)
    val1 = float(b1 ** 2 - 4 * a1 * c1)
    sq1 = np.sqrt(np.maximum(val1, 0))
    r1  = (b1 + sq1) / 2

    a2  = 4
    b2  = 2 * (height + width)
    c2  = (1 - min_overlap) * width * height
    val2 = float(b2 ** 2 - 4 * a2 * c2)
    sq2 = np.sqrt(np.maximum(val2, 0))
    r2  = (b2 + sq2) / 2

    a3  = 4 * min_overlap
    b3  = -2 * min_overlap * (height + width)
    c3  = (min_overlap - 1) * width * height
        
    val3 = float(b3 ** 2 - 4 * a3 * c3)
    sq3 = np.sqrt(np.maximum(val3, 0))
    r3  = (b3 + sq3) / 2
    
    return min(r1, r2, r3)
class VISOCenterNetDataset(VISOSequenceDataset):
    def __init__(self, img_dir, ann_file, sequence_length=5, transform=None, output_res=128):

        super().__init__(img_dir, ann_file, sequence_length, transform)
        self.output_res = output_res
        self.num_classes = 1 

    def draw_gaussian(self, heatmap, center, radius, k=1):
        """Draws a 2D Gaussian kernel on the heatmap."""
        diameter = 2 * radius + 1
        gaussian = self.gaussian2D((diameter, diameter), sigma=diameter / 6)
        
        x, y = int(center[0]), int(center[1])
        height, width = heatmap.shape[0:2]
        
        left, right = min(x, radius), min(width - x - 1, radius)
        top, bottom = min(y, radius), min(height - y - 1, radius)

        masked_heatmap = heatmap[y - top:y + bottom + 1, x - left:x + right + 1]
        masked_gaussian = gaussian[radius - top:radius + bottom + 1, radius - left:radius + right + 1]
        if min(masked_gaussian.shape) > 0 and min(masked_heatmap.shape) > 0:
            np.maximum(masked_heatmap, masked_gaussian * k, out=masked_heatmap)
        return heatmap

    def gaussian2D(self, shape, sigma=1):
        m, n = [(ss - 1.) / 2. for ss in shape]
        y, x = np.ogrid[-m:m + 1, -n:n + 1]
        h = np.exp(-(x * x + y * y) / (2 * sigma * sigma))
        h[h < np.finfo(h.dtype).eps * h.max()] = 0
        return h
    def __getitem__(self, idx):
    
        sequence_tensor, raw_target = super().__getitem__(idx)
        bboxes = raw_target["bboxes"]
        

        hm = np.zeros((self.num_classes, self.output_res, self.output_res), dtype=np.float32)
        wh = np.zeros((2, self.output_res, self.output_res), dtype=np.float32)
        reg = np.zeros((2, self.output_res, self.output_res), dtype=np.float32)
        reg_mask = np.zeros((1, self.output_res, self.output_res), dtype=np.float32)

        input_res = 512 
        scale = input_res / self.output_res

        for i in range(len(bboxes)):
            bbox = bboxes[i]
            x1 = np.clip(bbox[0], 0, input_res - 1)
            y1 = np.clip(bbox[1], 0, input_res - 1)
            x2 = np.clip(bbox[2], 0, input_res - 1)
            y2 = np.clip(bbox[3], 0, input_res - 1)
            w, h = (x2 - x1) / scale, (y2 - y1) / scale
            

            if h > 0.1 and w > 0.1:

                ct = np.array([(x1 + x2) / (2 * scale), (y1 + y2) / (2 * scale)], dtype=np.float32)
                ct_int = ct.astype(np.int32)
                if ct_int[0] < self.output_res and ct_int[1] < self.output_res:

                    radius = gaussian_radius((h, w), min_overlap=0.7)
                    radius = max(1, int(radius))
                    
                    self.draw_gaussian(hm[0], ct_int, radius)
                     
                    wh[0, ct_int[1], ct_int[0]] = w
                    wh[1, ct_int[1], ct_int[0]] = h
                    
                    reg[0, ct_int[1], ct_int[0]] = ct[0] - ct_int[0]
                    reg[1, ct_int[1], ct_int[0]] = ct[1] - ct_int[1]
                    
                    reg_mask[0, ct_int[1], ct_int[0]] = 1

        target = {
            "hm": torch.from_numpy(hm),
            "wh": torch.from_numpy(wh),
            "reg": torch.from_numpy(reg),
            "reg_mask": torch.from_numpy(reg_mask),
            "raw_bboxes": bboxes 
        }

        return sequence_tensor, target

def collate_fn_centrenet(batch):
    sequences = torch.stack([item[0] for item in batch], dim=0)
    
    targets = {
        "hm": torch.stack([item[1]["hm"] for item in batch], dim=0),
        "wh": torch.stack([item[1]["wh"] for item in batch], dim=0),
        "reg": torch.stack([item[1]["reg"] for item in batch], dim=0),
        "reg_mask": torch.stack([item[1]["reg_mask"] for item in batch], dim=0),
        "raw_bboxes": [item[1]["raw_bboxes"] for item in batch]
    }
    return sequences, targets



viso_dataset = VISOCenterNetDataset(
    img_dir=IMG_DIR,
    ann_file=ANN_FILE,
    sequence_length=5,
    transform=transform,
    output_res=128 # 512 / 4
)

data_loader = DataLoader(
    viso_dataset, 
    batch_size=2, 
    shuffle=True, 
    num_workers=0, 
    collate_fn=collate_fn_centrenet
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

class CenterNetHead(nn.Module):
    def __init__(self, in_channels, num_classes=1):
        super(CenterNetHead, self).__init__()
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, num_classes, kernel_size=1)
        )
        self.wh_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, 2, kernel_size=1)
        )
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, 2, kernel_size=1)
        )
        self.cls_head[-1].bias.data.fill_(-2.19)

    def forward(self, x):
        hm = torch.sigmoid(self.cls_head(x)) 
        wh = self.wh_head(x)
        reg = self.reg_head(x)
        return {'hm': hm, 'wh': wh, 'reg': reg}

def focal_loss(pred, gt):
    pos_inds = gt.eq(1).float()
    neg_inds = gt.lt(1).float()
    
    neg_weights = torch.pow(1 - gt, 6)
    
    loss = 0
    pred = torch.clamp(pred, 1e-4, 1 - 1e-4) # Prevent log(0)
    
    pos_loss = torch.log(pred) * torch.pow(1 - pred, 2) * pos_inds
    neg_loss = torch.log(1 - pred) * torch.pow(pred, 2) * neg_weights * neg_inds
    
    num_pos = pos_inds.float().sum()
    pos_loss = pos_loss.sum()
    neg_loss = neg_loss.sum()
    
    if num_pos == 0:
        loss = loss - neg_loss
    else:
        loss = loss - (pos_loss + neg_loss) / num_pos
    return loss

class MICPLLoss(nn.Module):
    def __init__(self, lambda_size=0.1, lambda_off=1.0):
        super(MICPLLoss, self).__init__()
        self.lambda_size = lambda_size
        self.lambda_off = lambda_off

    def forward(self, preds, targets):
        hm_pred = preds['hm']
        wh_pred = preds['wh']
        reg_pred = preds['reg']
        
        hm_gt, wh_gt, reg_gt, reg_mask = targets['hm'], targets['wh'], targets['reg'], targets['reg_mask']
        l_key = focal_loss(hm_pred, hm_gt)
        
        l_size = F.l1_loss(wh_pred * reg_mask, wh_gt * reg_mask, reduction='sum') / (reg_mask.sum() + 1e-4)
    
        l_off = F.l1_loss(reg_pred * reg_mask, reg_gt * reg_mask, reduction='sum') / (reg_mask.sum() + 1e-4)
        
        total_loss = l_key + (self.lambda_size * l_size) + (self.lambda_off * l_off)
        return total_loss, l_key, l_size, l_off

### Pipeline

In [ ]:
from torchvision.ops import nms

def decode_predictions(hm, wh, reg, K=100, nms_threshold=0.3):
    batch, cat, height, width = hm.size()
    
    hmax = F.max_pool2d(hm, kernel_size=3, stride=1, padding=1)
    keep = (hmax == hm).float()
    hm = hm * keep
    
    hm_flat = hm.view(batch, -1)
    scores, inds = torch.topk(hm_flat, K)
    
    ys = (inds // width).float()
    xs = (inds % width).float()
    
    reg = reg.view(batch, 2, -1)
    xs = xs + reg[:, 0, :].gather(1, inds)
    ys = ys + reg[:, 1, :].gather(1, inds)
    
    wh = wh.view(batch, 2, -1)
    ws = wh[:, 0, :].gather(1, inds)
    hs = wh[:, 1, :].gather(1, inds)
    
    bboxes = torch.stack([
        xs - ws / 2, ys - hs / 2,
        xs + ws / 2, ys + hs / 2
    ], dim=-1)
    

    out_boxes, out_scores = [], []
    for i in range(batch):
        keep_idx = nms(bboxes[i], scores[i], iou_threshold=nms_threshold)
        out_boxes.append(bboxes[i][keep_idx])
        out_scores.append(scores[i][keep_idx])
    
    return out_boxes, out_scores 

def calculate_iou(box1, box2):
    """Calculates IoU between two bounding boxes. Handles Tensors and Numpy."""

    if torch.is_tensor(box1): box1 = box1.detach().cpu().numpy()
    if torch.is_tensor(box2): box2 = box2.detach().cpu().numpy()
    
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    return inter_area / (box1_area + box2_area - inter_area + 1e-6)

def calculate_metrics(pred_boxes, pred_scores, gt_boxes, iou_threshold=0.5, score_threshold=0.1):
    """Calculates TP, FP, FN with safety checks for empty detections."""
    tp, fp, fn = 0, 0, 0
    
    mask = pred_scores > score_threshold
    valid_preds = pred_boxes[mask]
    
    if len(valid_preds) == 0:
        return 0, 0, len(gt_boxes)
    
    matched_gt = set()
    for pred in valid_preds:
        best_iou = 0
        best_gt_idx = -1
        
        for idx, gt in enumerate(gt_boxes):
            if idx in matched_gt: continue
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx
                
        if best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1
            
    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn

In [ ]:
from tqdm import tqdm
import time

def train_epoch(model, head, dataloader, optimizer, criterion, device):
    
    model.train()
    head.train()
    total_loss = 0
    
    for sequences, targets in tqdm(dataloader, desc="Training"):
        sequences = sequences.to(device)
        targets_gpu = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in targets.items()
        }
        
        optimizer.zero_grad()
        
        # Forward pass
        fused_features = model(sequences) 
        
        final_features = fused_features[:, :, -1, :, :] 
        preds = head(final_features)
        
        loss, l_key, l_size, l_off = criterion(preds, targets_gpu)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=35)
        optimizer.step()
        total_loss += loss.item()

        del sequences, targets_gpu, fused_features, final_features, preds, loss
        torch.cuda.empty_cache()

    return total_loss / len(dataloader)
@torch.no_grad()
def evaluate(model, head, dataloader, device):
    model.eval()
    head.eval()
    
    total_tp, total_fp, total_fn = 0, 0, 0
    start_time = time.time()
    num_frames = 0
    stride = 4 
     
    for sequences, targets in tqdm(dataloader, desc="Evaluating"):
        sequences = sequences.to(device)
        num_frames += sequences.size(0)
        
        vision_features = model(sequences)
        final_features = vision_features[:, :, -1, :, :]
        preds = head(final_features)
        
        pred_bboxes, pred_scores = decode_predictions(preds['hm'], preds['wh'], preds['reg'])
        
        for i in range(sequences.size(0)):
            pb = pred_bboxes[i].clone()
            pb[:, [0, 2]] *= stride
            pb[:, [1, 3]] *= stride
            
            gt_boxes = targets["raw_bboxes"][i]
            

            tp, fp, fn = calculate_metrics(pb, pred_scores[i], gt_boxes, score_threshold=0.5)
            total_tp += tp
            total_fp += fp
            total_fn += fn

    end_time = time.time()
    fps = num_frames / (end_time - start_time)
    
    precision = total_tp / (total_tp + total_fp + 1e-6)
    recall = total_tp / (total_tp + total_fn + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    
    print(f"Eval Results -> Pr: {precision:.4f} | Re: {recall:.4f} | F1: {f1:.4f} | FPS: {fps:.1f}")
    return f1, precision, recall

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_predictions(image_tensor, bboxes, scores, score_threshold=0.3):
    
    img = image_tensor.cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)
    
    valid_indices = scores > score_threshold
    valid_boxes = bboxes[valid_indices]
    
    for box in valid_boxes:
        x1, y1, x2, y2 = box.cpu().numpy()
        width = x2 - x1
        height = y2 - y1
        
        
        rect = patches.Rectangle(
            (x1, y1), width, height, 
            linewidth=1.5, edgecolor='blue', facecolor='none'
        )
        ax.add_patch(rect)
        
    plt.axis('off')
    plt.show()


In [ ]:
import torch
import torch.nn as nn
import os

import torchvision.models as models

class SimpleFeatureExtractor(nn.Module):
    def __init__(self, out_channels=64):
        super().__init__()
       
        resnet = models.resnet18(pretrained=True)
        
        
        self.net = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1
        )
        
    def forward(self, x):
        return self.net(x)


if __name__ == "__main__":

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    NUM_GPUS = torch.cuda.device_count()
    NUM_EPOCHS = 20           
    BATCH_SIZE = 4 * max(NUM_GPUS, 1)            
    LEARNING_RATE = 1.25e-4   
    CHANNELS = 64
    SEQ_LEN = 5               
    NUM_CLASSES = 1           
    
    print(f"Using device: {DEVICE}")

    
    print(f"Using device: {DEVICE} | GPUs available: {NUM_GPUS}")
    IMG_DIR_val = './VISO/coco/plane/val2017'
    ANN_FILE_val = './VISO/coco/plane/Annotations/instances_val2017.json'
    viso_dataset_val = VISOCenterNetDataset(
        img_dir=IMG_DIR_val,
        ann_file=ANN_FILE_val,
        sequence_length=5,
        transform=transform,
        output_res=128 
    )
    
    
    train_loader = DataLoader(viso_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn_centrenet)
    val_loader = DataLoader(viso_dataset_val, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn_centrenet)

    backbone = DLA34FeatureExtractor(out_channels=CHANNELS, pretrained=True)
    raft     = RAFTFlowEstimator()                        # pretrained, frozen
    model    = SmallObjectDetector(backbone, raft, channels=CHANNELS, T=SEQ_LEN, num_heads=4).to(DEVICE)
    head = CenterNetHead(in_channels=CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)
    # checkpoint = torch.load(
    #     "/kaggle/input/models/arghyadeep192/model-withoutbifpn-20/pytorch/default/1/best_micpl_model (1).pth",
    #     map_location=DEVICE
    # )

    # missing, unexpected = model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    # head.load_state_dict(checkpoint['head_state_dict'], strict=False)

    if NUM_GPUS > 1:
        print(f"Wrapping model and head in DataParallel across {NUM_GPUS} GPUs")
        model = nn.DataParallel(model)
        head  = nn.DataParallel(head)
        torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False)

    criterion = MICPLLoss(lambda_size=1.0, lambda_off=1.0)
    
    _micpl_params    = list(model.module.micpl.parameters()) if NUM_GPUS > 1 else list(model.micpl.parameters())
    _backbone_params = list(model.module.backbone.parameters()) if NUM_GPUS > 1 else list(model.backbone.parameters())
    _raft_params     = []   
    optimizer = torch.optim.Adam([
        {'params': _micpl_params,    'lr': LEARNING_RATE},
        {'params': _backbone_params, 'lr': 0.0, 'name': 'backbone'},
    ], lr=LEARNING_RATE)
    print("DLA-34 + BiFPN backbone params :", sum(p.numel() for p in backbone.parameters()))
    print("RAFT params (frozen)            :", sum(p.numel() for p in raft.parameters()))
    print("Full model params               :", sum(p.numel() for p in model.parameters()))

    print("Detection head params:",
          sum(p.numel() for p in head.parameters()))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=20,      
        eta_min=1e-6   
    )
    f1_scores,pr_scores,recall_scores=[],[],[]
    # --- Training Loop ---
    best_f1 = 0.0
    save_dir = "./saved_models"
    os.makedirs(save_dir, exist_ok=True)

    print("Starting training...")
    for epoch in range(NUM_EPOCHS):
        if epoch == 10:
            if NUM_GPUS > 1:
                model.module.backbone.unfreeze_backbone()
            else:
                model.backbone.unfreeze_backbone()
            for pg in optimizer.param_groups:
                if pg.get('name') == 'backbone':
                    pg['lr'] = 1e-5
                    break
        print(f"\n--- Epoch [{epoch+1}/{NUM_EPOCHS}] ---")
        
        avg_train_loss = train_epoch(model, head, train_loader, optimizer, criterion, DEVICE)
        print(f"Average Training Loss: {avg_train_loss:.4f}")
        
        torch.cuda.empty_cache()
        f1, precision, recall = evaluate(model, head, val_loader, DEVICE)
        torch.cuda.empty_cache()
        f1_scores.append(f1)
        pr_scores.append(precision)
        recall_scores.append(recall)
        

        scheduler.step() 
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current Learning Rate: {current_lr}")

        if f1 > best_f1:
            best_f1 = f1
            save_path = os.path.join(save_dir, "best_micpl_model.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.module.state_dict() if NUM_GPUS > 1 else model.state_dict(),
                'head_state_dict':  head.module.state_dict()  if NUM_GPUS > 1 else head.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
            }, save_path)
            print(f"New best model saved with F1: {best_f1:.4f}")

    print("Training complete! Best F1 Score:", best_f1)

In [ ]:
import matplotlib.pyplot as plt


epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs, f1_scores,      label='F1',        color='royalblue',  linewidth=2, marker='o', markersize=3)
plt.plot(epochs, pr_scores,      label='Precision',  color='darkorange', linewidth=2, marker='s', markersize=3)
plt.plot(epochs, recall_scores,  label='Recall',     color='seagreen',   linewidth=2, marker='^', markersize=3)


for milestone in [35, 45]:
    plt.axvline(x=milestone, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    plt.text(milestone + 0.3, 0.02, f'LR drop (ep {milestone})', 
             fontsize=8, color='gray', rotation=90, va='bottom')


best_epoch = f1_scores.index(max(f1_scores)) + 1
plt.axvline(x=best_epoch, color='royalblue', linestyle=':', linewidth=1.2, alpha=0.6)
plt.text(best_epoch + 0.3, max(f1_scores) - 0.05,
         f'Best F1: {max(f1_scores):.4f} (ep {best_epoch})',
         fontsize=8, color='royalblue')

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('F1 / Precision / Recall vs Epochs', fontsize=14)
plt.legend(fontsize=11)
plt.ylim(0, 1.05)
plt.xlim(1, NUM_EPOCHS)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./saved_models/metrics_plot.png', dpi=150)
plt.show()
print("Plot saved to ./saved_models/metrics_plot.png")

In [ ]:
import matplotlib.pyplot as plt
import cv2

def unnormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img * std[:, None, None] + mean[:, None, None]
    return img

def visualize_sample(model, head, val_loader, device):
    model.eval()
    head.eval()

    sequences, targets = next(iter(val_loader))
    sequences = sequences.to(device)

    with torch.no_grad():
        fused = model(sequences)
        final_feat = fused[:, :, -1, :, :]
        preds = head(final_feat)

    bboxes, scores = decode_predictions(
        preds['hm'], preds['wh'], preds['reg'], K=100
    )
    seq = sequences[0].cpu().numpy()
    gt_boxes = targets["raw_bboxes"][0]

    pred_boxes = bboxes[0].cpu().numpy()
    pred_scores = scores[0].cpu().numpy()
    img = seq[:, -1]
    img = unnormalize(img).transpose(1, 2, 0)
    img = np.clip(img, 0, 1)
    img = (img * 255).astype(np.uint8)
    img = cv2.resize(img, (512, 512))

    for box in gt_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    scale = 512 / 128  

    for i in range(len(pred_boxes)):
        score = pred_scores[i]

        if score < 0.3:
            continue

        x1, y1, x2, y2 = pred_boxes[i]

        x1 = int(x1 * scale)
        y1 = int(y1 * scale)
        x2 = int(x2 * scale)
        y2 = int(y2 * scale)

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(img, f"{score:.2f}", (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title("Green = GT | Red = Pred")
    plt.axis('off')
    plt.show()

In [ ]:
visualize_sample(model,head,val_loader,DEVICE)

### Testing

In [ ]:
IMG_DIR_test = './VISO/coco/plane/test2017'
ANN_FILE_test = './VISO/coco/plane/Annotations/instances_test2017.json'
viso_dataset_test = VISOCenterNetDataset(
    img_dir=IMG_DIR_test,
    ann_file=ANN_FILE_test,
    sequence_length=5,
    transform=transform,
    output_res=128 
)
test_loader = DataLoader(viso_dataset_test, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn_centrenet)

In [ ]:
checkpoint = torch.load("./saved_models/best_micpl_model.pth", map_location=DEVICE)
backbone = DLA34FeatureExtractor(out_channels=CHANNELS, pretrained=False)
raft     = RAFTFlowEstimator()
model    = SmallObjectDetector(backbone, raft, channels=CHANNELS, T=SEQ_LEN, num_heads=4).to(DEVICE)
head = CenterNetHead(in_channels=CHANNELS, num_classes=NUM_CLASSES).to(DEVICE)

model.load_state_dict(checkpoint['model_state_dict'])
head.load_state_dict(checkpoint['head_state_dict'])

model.eval()
head.eval()

print(f"Loaded model from epoch {checkpoint['epoch']} with best F1: {checkpoint['best_f1']:.4f}")

In [ ]:
f1, precision, recall = evaluate(model, head, test_loader, DEVICE)